# X-OCR: train the X-specific UI perception model (Colab entry point)

Pure-vision detector: screenshot → YOLO11 → UI element boxes/roles. No DOM/Playwright/OmniParser at runtime.

**Setup**: run `scripts/make_bundle.sh` locally to produce `x-ocr-bundle.zip` (code + dataset), upload it to Colab (or Drive), then Run All.
GPU: T4 is enough (nano/small models @1280).

In [ ]:
# 1) install dependencies
%pip -q install ultralytics onnx onnxsim pyyaml
import torch, ultralytics
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('ultralytics', ultralytics.__version__)

In [ ]:
# 2) fetch the bundle (upload x-ocr-bundle.zip via the Files pane, or point to Drive)
import os, zipfile, glob
BUNDLE = '/content/x-ocr-bundle.zip'
if not os.path.exists(BUNDLE):
    from google.colab import files
    print('upload x-ocr-bundle.zip')
    up = files.upload()
    BUNDLE = list(up.keys())[0]
!rm -rf /content/x-ocr && mkdir -p /content/x-ocr
with zipfile.ZipFile(BUNDLE) as z:
    z.extractall('/content/x-ocr')
os.chdir('/content/x-ocr')
print(os.listdir('.'))

In [ ]:
# 3) validate dataset: counts, splits, zero-instance classes, corrupt images
import yaml, json
from pathlib import Path
from PIL import Image

d = yaml.safe_load(Path('data/yolo/data.yaml').read_text())
meta = json.loads(Path('data/yolo/export_meta.json').read_text())
print('splits:', meta['splits'])
print('instances/class:', list(meta['instances_per_class'].items())[:10], '...')
zero = [c for c, n in meta['instances_per_class'].items() if n == 0]
print('zero-instance classes:', zero or 'none')
bad = 0
for p in Path('data/yolo/images').rglob('*.png'):
    try:
        with Image.open(p) as im:
            im.verify()
    except Exception:
        bad += 1
print('corrupt images:', bad)
assert bad == 0

In [ ]:
# 4) train (nano baseline; switch --model yolo11s.pt / --imgsz for variants)
!python training/train_yolo.py --data data/yolo/data.yaml --model yolo11n.pt \
    --imgsz 1280 --epochs 60 --batch auto --exp exp001_baseline_nano --out experiments

In [ ]:
# 5) evaluate on the frozen test split (mAP, per-class, Key Interactive Element Recall)
!python evaluation/eval_frozen.py --data data/yolo/data.yaml \
    --weights experiments/exp001_baseline_nano/run/weights/best.pt --imgsz 1280

In [ ]:
# 6) results: metrics + plots
import json, glob
from IPython.display import Image as IPyImage, display
print(json.dumps(json.load(open('experiments/exp001_baseline_nano/metrics.json')), indent=1)[:1200])
for p in glob.glob('experiments/exp001_baseline_nano/run/*.png')[:6]:
    print(p); display(IPyImage(p, width=600))

In [ ]:
# 7) save checkpoints back (Drive optional)
import shutil
shutil.make_archive('/content/xocr_exp001_artifacts', 'zip', 'experiments/exp001_baseline_nano')
print('artifacts →', '/content/xocr_exp001_artifacts.zip')
try:
    from google.colab import drive
    drive.mount('/content/drive')
    shutil.copy('/content/xocr_exp001_artifacts.zip', '/content/drive/MyDrive/')
except Exception as e:
    print('drive skipped:', e)